# AdCanvas 메인 노트북 (배포용)

사진 한 장 → ① Segmentation → ② VLM(요청 해석) → ③ 배경 생성 → ④ 원본 음식 재합성 → 광고 이미지 + 문구

* 각 단계는 앞 단계 결과를 정해진 형식(`nukki_out` → `vlm_out` → `final_out`)으로 받아서 씀
* 평가(점수, 비교 실험)는 서브 노트북(`AdCanvas_eval.ipynb`)에서 함
* 런타임: GPU(T4). 위에서부터 차례로 실행하면 5번에서 웹 서비스용 서버까지 켜짐

# 0. 설치
* `pillow<13` 고정: Pillow 13에서 이미지 생성 단계의 일부 함수가 동작하지 않음
* 설치 중 Pillow 버전이 바뀌면 세션이 다시 시작됨 → 다시 처음부터 실행

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif "pillow<13" qwen-vl-utils "peft==0.14.0"
!pip -q install --upgrade openai

# 설치 중 Pillow 버전이 바뀌면 세션 다시 시작 (메모리와 디스크 버전이 섞이는 것 방지)
import PIL, importlib.metadata as _md
if PIL.__version__ != _md.version('pillow'):
    print(f'Pillow {PIL.__version__} → {_md.version("pillow")} 로 바뀌어서 세션을 다시 시작해요. 끝나면 "모두 실행"을 한 번 더 눌러 주세요.')
    import os; os.kill(os.getpid(), 9)

# 1. Segmentation

**사진 → 그릇 후보 → 그릇 고르기 → 음식 마스크 + 투명 배경 cutout**

| 단계 | 모델 | 역할 |
|---|---|---|
| 그릇 찾기 | Grounding DINO tiny | 글자(plate, bowl …)로 그릇·음식 박스 찾기 |
| 그릇별 마스크 | SAM 2.1 hiera-small + 파인튜닝 디코더 | 박스 하나 → 그릇 + 음식 + 걸친 수저를 통째로 |
| 후처리 | 직접 작성 | 조각 정리, 구멍 채우기, 중복 그릇 제거 |
| 그릇 고르기 | 직접 작성 | 1-6: 직접 고름 / 1-9(서버): 메인 그릇 자동 선택 |

* 파인튜닝 가중치: `AdCanvas_nukki/weights/sam2_ft.pt` (없으면 원래 SAM으로 동작)
* 결과 마스크: **흰색(255) = 음식(그릇 포함)**, 원본 사진과 같은 크기

## 1-1. 설정
* `NUKKI_WEIGHTS`: 파인튜닝 가중치 위치 (팀 드라이브 공유 폴더 기준)
* `OUT_DIR`: 결과 저장 위치 (다음 단계가 여기서 읽음)

In [ ]:
import os, io, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from google.colab import drive
drive.mount('/content/drive')

NUKKI_BASE = '/content/drive/MyDrive/AdCanvas_nukki'
NUKKI_WEIGHTS = f'{NUKKI_BASE}/weights/sam2_ft.pt'
OUT_DIR = '/content/adcanvas_out'
os.makedirs(OUT_DIR, exist_ok=True)
print('파인튜닝 가중치:', '있음' if os.path.exists(NUKKI_WEIGHTS) else '없음 (원래 SAM 사용)')

## 1-2. Segmentation 함수
실행만 하면 됨. 바깥에서 쓰는 함수:

| 함수 | 입력 → 출력 |
|---|---|
| `load()` | 모델 불러오기 (한 번) |
| `candidates(img)` | 사진 → 그릇 후보 목록 `[{'id','label','score','kind','dish','box','mask'}]` |
| `merge(img, cands, ids)` | 고른 후보만 합친 마스크 (ids=None이면 기본 선택 전부) |
| `segment(img)` | 사진 → 마스크 (자동, 기본 선택 전부) |
| `refine(img, box, points, labels)` | 사용자 박스·클릭 보정 → 그릇 하나 마스크 |
| `cutout(img, mask)` | 투명 배경 cutout (RGBA) |

In [ ]:
import os

import cv2
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy import ndimage

SAM_MODEL = 'facebook/sam2.1-hiera-small'
GD_ID = 'IDEA-Research/grounding-dino-tiny'
MAX_SIDE = 2048            # 이보다 큰 사진은 줄여서 처리하고, 마스크는 원래 크기로 되돌림

# Segmentation-SAM 파이프라인과 같은 값 (test 20장 IoU 0.906, BF 0.827)
DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3, nms_iou=0.6, max_area=0.9, food_max_area=0.5,
)
CAND = dict(
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,   # 수저의 20% 이상이 그릇 외곽 안에 있으면 걸친 것 (원래 SAM일 때만 합침)
    part_min=0.1,          # 마스크 조각 중 가장 큰 조각의 10% 이상만 남김
    solidity=0.9,          # 원래 SAM일 때: 그릇이 볼록 외곽의 90% 미만이면 외곽으로 채움
    inside_drop=0.9,       # 다른 그릇 안에 90% 이상 들어간 후보는 중복으로 제거
    sam_chunk=4,           # SAM을 박스 몇 개씩 돌릴지 (메모리 절약)
)

_m = {}   # 불러온 모델: device, gd_proc, gd, sam, finetuned


# ---------------- 불러오기 ----------------
def _device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def load(weights=NUKKI_WEIGHTS):
    """모델 불러오기 (서버 켤 때 한 번). 반환: 파인튜닝 가중치를 썼는지"""
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    try:
        from pillow_heif import register_heif_opener
        register_heif_opener()   # 아이폰 HEIC
    except ImportError:
        pass
    dev = _device()
    _m['device'] = dev
    _m['gd_proc'] = AutoProcessor.from_pretrained(GD_ID)
    _m['gd'] = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(dev).eval()
    sam = SAM2ImagePredictor.from_pretrained(SAM_MODEL, device=dev)
    _m['finetuned'] = bool(weights) and os.path.exists(weights)
    if _m['finetuned']:
        res = sam.model.load_state_dict(torch.load(weights, map_location=dev), strict=False)
        assert not res.missing_keys and not res.unexpected_keys, 'sam2_ft.pt와 SAM_MODEL이 맞지 않음'
    _m['sam'] = sam
    _m['sam_img'] = None   # set_image 한 사진 (클릭 보정 때 다시 계산하지 않으려고)
    return _m['finetuned']


def _ready():
    if 'sam' not in _m:
        load()


def _prep(img):
    """PIL 사진 → (처리용 RGB 사진, 원래 크기). EXIF 회전 반영, 긴 변 MAX_SIDE로"""
    img = ImageOps.exif_transpose(img).convert('RGB')
    size = img.size
    if max(size) > MAX_SIDE:
        img = img.copy()
        img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    return img, size


def _to_size(m, size):
    """bool 마스크 → 원래 크기의 흑백 PIL (L, 0/255)"""
    out = Image.fromarray((m * 255).astype(np.uint8))
    return out if out.size == size else out.resize(size, Image.NEAREST)


def _set_image(img_np):
    key = (img_np.shape, int(img_np[::97, ::89].sum()))
    if _m['sam_img'] != key:
        _m['sam'].set_image(img_np)
        _m['sam_img'] = key


# ---------------- DINO ----------------
def _box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    from torchvision.ops import nms
    C, proc, gd = DINO_CFG, _m['gd_proc'], _m['gd']
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(_m['device'])
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and _box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]


# ---------------- 마스크 도우미 ----------------
def _box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def _local(fn, m, pad=0):
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull(m):
    def f(c):
        out = np.zeros(c.shape, np.uint8)
        cv2.fillConvexPoly(out, cv2.convexHull(cv2.findNonZero(c.astype(np.uint8))), 1)
        return out.astype(bool)
    return _local(f, m)


def _close_fill(m):
    return _local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def _keep_parts(c, box):
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(_box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def _crop(m, box, fn, pad=4):
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def _pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = _box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


def _clean_dish(m, box):
    out = _crop(m, box, lambda c, b: ndimage.binary_fill_holes(ndimage.binary_closing(_keep_parts(c, b), iterations=3)))
    if not out.any():
        return out
    h = _hull(out) & (_box_mask(m.shape, box, 0.05) | out)
    return h if out.sum() < CAND['solidity'] * h.sum() else out


# ---------------- SAM + 후처리 ----------------
@torch.no_grad()
def _sam_masks(img_np, boxes, kinds):
    """파인튜닝 SAM은 마스크 1개, 원래 SAM은 3개 중 전체 물체를 고름"""
    _set_image(img_np)
    H, W = img_np.shape[:2]
    multi = not _m['finetuned']
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = _m['sam'].predict(box=b, multimask_output=multi)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multi:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            out.append(m[k, int(sc[k].argmax())] if kinds[s0 + k] == 'other' else _pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def _tidy(cands, light):
    for c in cands:
        if c['dish'] and c['kind'] != 'food':
            c['mask'] = _close_fill(_crop(c['mask'], c['box'], _keep_parts, pad=0)) if light else _clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c['kind'] != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if 'dup_of' not in big and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        union |= d['mask']
    for f in [c for c in cands if c['kind'] == 'food']:
        f['mask'] = _close_fill(_crop(f['mask'], f['box'], _keep_parts, pad=0))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if u['label'] in CAND['utensil_words'] and u['mask'].any()]
    if utensils and not light:
        hulls = {c['id']: _hull(c['mask']) for c in cands if c['dish'] and 'dup_of' not in c and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = _close_fill(d['mask'] | u['mask'])
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


# ---------------- 바깥에서 쓰는 함수 ----------------
def candidates(img):
    """사진 → 그릇 후보 목록. 박스는 원래 사진 좌표, mask는 원래 크기의 흑백 PIL
    [{'id', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택), 'box'[x0,y0,x1,y1], 'mask'}]
    그릇을 하나도 못 찾으면 [] (서비스에서는 사용자가 직접 박스를 그리게 → refine)"""
    _ready()
    work, size = _prep(img)
    found = dino_boxes(work)
    if not any(f['dish'] for f in found):
        return []
    masks = _sam_masks(np.array(work), [f['box'] for f in found], [f['kind'] for f in found])
    for i, (f, m) in enumerate(zip(found, masks)):
        f.update(id=i, mask=m)
    cands = _tidy(found, light=_m['finetuned'])
    s = size[0] / work.size[0]
    return [{'id': c['id'], 'label': c['label'], 'score': round(c['score'], 3), 'kind': c['kind'], 'dish': c['dish'],
             'box': [round(v * s, 1) for v in c['box']], 'mask': _to_size(c['mask'], size)} for c in cands]


def merge(img, cands, ids=None):
    """후보 중 고른 것만 합친 마스크 (ids=None이면 기본 선택된 그릇 전부)"""
    W, H = ImageOps.exif_transpose(img).size
    out = np.zeros((H, W), bool)
    for c in cands:
        if (c['id'] in ids) if ids is not None else c['dish']:
            out |= np.asarray(c['mask']) > 127
    return _to_size(out, (W, H))


def segment(img):
    """팀 공통 함수 모양 (KAN-5): RGB 사진 → 흑백 마스크 (L, 사진과 같은 크기, 255 = 음식+그릇)"""
    return merge(img, candidates(img))


@torch.no_grad()
def refine(img, box, points=None, labels=None):
    """클릭 보정: 박스(원래 사진 좌표) + 클릭 점(1 포함, 0 제외) → 그 그릇 하나의 마스크 (L, 원래 크기)"""
    _ready()
    work, size = _prep(img)
    s = work.size[0] / size[0]
    _set_image(np.array(work))
    kw = dict(box=np.array([v * s for v in box], np.float32), multimask_output=False)
    if points:
        kw.update(point_coords=np.array(points, np.float32) * s, point_labels=np.array(labels, np.int32))
    m, _, _ = _m['sam'].predict(**kw)
    return _to_size(np.asarray(m)[0] > 0.5, size)


def cutout(img, mask, crop=False):
    """원본 + 마스크 → 투명 배경 cutout (RGBA). crop=True면 음식 부분만 잘라냄"""
    rgb = ImageOps.exif_transpose(img).convert('RGB')
    out = rgb.copy()
    out.putalpha(mask.convert('L').resize(rgb.size, Image.NEAREST))
    return out.crop(out.getbbox()) if crop and out.getbbox() else out

## 1-3. 모델 불러오기

In [ ]:
t0 = time.time()
ft = load(NUKKI_WEIGHTS)
print(f"device: {_m['device']} | 파인튜닝 SAM: {ft} | {time.time() - t0:.1f}초")

## 1-4. 사진 넣기
* `IMG_PATH`에 경로를 넣거나, 비워 두면 업로드 창이 뜸

In [ ]:
IMG_PATH = f''
# 예: f'{NUKKI_BASE}/dataset/images/hwang/IMG_0405.HEIC'

if not IMG_PATH:
    from google.colab import files
    up = files.upload()
    IMG_PATH = f'/content/{next(iter(up))}'
img = ImageOps.exif_transpose(Image.open(IMG_PATH)).convert('RGB')
print(IMG_PATH, img.size)

## 1-5. 그릇 후보 찾기
* 초록 = 그릇(기본 선택), 주황 = 컵·수저 등(기본 선택 안 함)
* 그릇을 하나도 못 찾으면 1-7(직접 박스)로

In [ ]:
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def show_candidates(img, cands):
    base = np.array(img)
    over = base.copy()
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(base)
    for k, c in enumerate(cands):
        x0, y0, x1, y1 = c['box']
        col = 'lime' if c['dish'] else 'orange'
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=col, lw=2))
        ax[0].text(x0, y0, f"{c['id']}:{c['label']} {c['score']:.2f}", fontsize=8, backgroundcolor=col)
        if c['dish']:
            m = np.asarray(c['mask']) > 127
            over[m] = (over[m] * 0.45 + COLORS[k % len(COLORS)] * 0.55).astype(np.uint8)
    ax[0].set_title('candidates (green = dish)'); ax[1].imshow(over); ax[1].set_title('dish candidates')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()


t0 = time.time()
cands = candidates(img)
print(f"후보 {len(cands)}개, 그릇 {sum(c['dish'] for c in cands)}개 | {time.time() - t0:.1f}초")
print('그릇 id:', [c['id'] for c in cands if c['dish']])
if cands:
    show_candidates(img, cands)
else:
    print('그릇을 찾지 못했어요. 1-7에서 박스를 직접 지정하세요.')

## 1-6. 그릇 고르기 → 마스크, cutout
* `SELECT = None`: 찾은 그릇 전부 / `SELECT = [0, 2]`: 고른 id만
* 서버에서는 1-9의 자동 선택(메인 그릇 위주)을 사용

In [ ]:
SELECT = None

mask = merge(img, cands, SELECT) if cands else None

## 1-7. (선택) 직접 박스·클릭으로 보정
* 자동으로 못 찾았거나 고치고 싶을 때만. `BOX = [x0, y0, x1, y1]` (원본 사진 좌표)
* `POINTS`: 클릭 좌표, `LABELS`: 1 포함 / 0 제외

In [ ]:
BOX = None          # 예: [400, 900, 2600, 3300]
POINTS, LABELS = None, None

if BOX:
    m = refine(img, BOX, POINTS, LABELS)
    mask = m if mask is None else Image.fromarray(np.maximum(np.asarray(mask), np.asarray(m)))
assert mask is not None, '마스크가 없어요. 1-6 또는 1-7을 확인하세요.'

cut = cutout(img, mask)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img); ax[0].set_title('image')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('mask (white = food)')
ax[2].imshow(cut); ax[2].set_title('cutout')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()

## 1-8. 다음 단계(VLM)에 넘기기
* `OUT_DIR`에 파일로 저장하고, 같은 내용을 `nukki_out` 변수로도 넘김
* 서버용 `nukki()`(1-9)도 같은 형식을 돌려줌 (+ `name`: 요청마다 다른 파일 이름)

| 키 | 내용 |
|---|---|
| `image` | 원본 사진 경로 (RGB, EXIF 회전 반영) |
| `mask` | 음식 마스크 경로 (L, 흰색 = 음식, 원본과 같은 크기) |
| `cutout` | 투명 배경 cutout 경로 (RGBA, 원본과 같은 크기) |
| `cutout_crop` | 음식 부분만 잘라낸 cutout 경로 |
| `food_box` | 음식 영역 박스 `[x0, y0, x1, y1]` |
| `dishes` | 선택된 그릇 목록 (id, 이름, 점수, 박스) |

In [ ]:
def to_next_stage(img, mask, cands, select, out_dir=OUT_DIR, name='input'):
    """Segmentation 결과 → 다음 단계가 읽을 형식 (파일 + dict)"""
    paths = {k: f'{out_dir}/{name}_{k}.png' for k in ('image', 'mask', 'cutout', 'cutout_crop')}
    img.save(paths['image'])
    mask.save(paths['mask'])
    cut = cutout(img, mask)
    cut.save(paths['cutout'])
    cutout(img, mask, crop=True).save(paths['cutout_crop'])
    box = cut.getbbox()
    chosen = [c for c in cands if ((c['id'] in select) if select is not None else c['dish'])]
    out = dict(paths, size=list(img.size), food_box=list(box) if box else None,
               dishes=[{k: c[k] for k in ('id', 'label', 'score', 'box')} for c in chosen],
               finetuned=_m['finetuned'])
    json.dump(out, open(f'{out_dir}/{name}_nukki.json', 'w'), ensure_ascii=False, indent=1)
    return out


nukki_out = to_next_stage(img, mask, cands, SELECT)
print(json.dumps({k: v for k, v in nukki_out.items() if k != 'dishes'}, ensure_ascii=False, indent=1))
print('그릇', len(nukki_out['dishes']), '개 선택')

## 1-9. 서버용 함수 `nukki()`
* 서버(5번)에서 요청마다 부르는 함수. 사진 한 장 → `nukki_out` (1-8과 같은 형식)
* 그릇을 못 찾으면 `NukkiNotFound` → 서버는 422로 안내

**그릇 자동 선택**: 실제 식탁 사진에는 앞접시, 옆 사람 접시, 사진 끝에 잘린 그릇이 함께 찍혀 있어서,
찾은 그릇을 전부 쓰면 광고에 불필요한 그릇이 들어감. 그래서 광고할 그릇만 고름.
1. 그릇마다 점수: 넓을수록, 사진 가운데에 가까울수록 높음. 사진 끝에 잘린 그릇은 0.3배
2. 점수 1등(메인)은 항상 사용
3. 나머지는 메인 점수의 70% 이상이면서 사진 끝에 안 걸린 그릇만 같이 사용
4. 상품명에 세트·정식·한상·백반·코스 등이 있으면 상차림 전체가 상품이므로, 사진 끝에 안 걸린 그릇 전부 사용
* boxes(사용자 박스)를 주면 박스 안의 그릇만 사용. 찾은 그릇이 없으면 그 영역을 SAM으로 새로 땀

In [ ]:
import uuid

MAIN_ONLY = True    # 서버 기본: 메인 그릇만 (앞접시·옆 접시 제외)
MAIN_KEEP = 0.7     # 메인 점수의 70% 이상이면서 사진 가장자리에 안 걸친 그릇은 같이 (세트 메뉴)
EDGE = 0.01         # 박스가 사진 끝에서 1% 안쪽이면 '잘린 그릇'으로 봄
BOX_INSIDE = 0.6    # 그릇 박스의 60% 이상이 사용자 박스 안에 있으면 '박스 안의 그릇'으로 봄
SET_WORDS = ('세트', '정식', '한상', '백반', '코스', '상차림', '한정식', '오마카세', 'set', 'course')
# 상품명에 위 단어가 있으면 상차림 전체가 상품 → 사진 끝에 안 걸린 그릇 전부


def is_set_menu(product_name):
    name = (product_name or '').replace(' ', '').lower()
    return any(w in name for w in SET_WORDS)


class NukkiNotFound(Exception):
    """그릇·음식을 찾지 못함 → 서버는 '음식을 직접 표시해 주세요' 안내 (박스 보정으로 다시 요청)"""


def _as_image(src):
    """PIL 이미지 / 파일 경로 / bytes(업로드 파일) → RGB PIL (EXIF 회전 반영)"""
    if isinstance(src, (bytes, bytearray)):
        src = Image.open(io.BytesIO(src))
    elif isinstance(src, (str, os.PathLike)):
        src = Image.open(src)
    return ImageOps.exif_transpose(src).convert('RGB')


def pick_main(cands, size, keep=MAIN_KEEP, set_menu=False):
    """기본 선택 그릇 중 메인 그릇 id 목록. 점수 = 넓이 × 가운데에 가까울수록 ↑ × (가장자리에 잘렸으면 0.3배)
    set_menu=True면 메인 + 사진 끝에 안 걸린 그릇 전부"""
    W, H = size
    def info(c):
        x0, y0, x1, y1 = c['box']
        area = float((np.asarray(c['mask']) > 127).sum())
        d = np.hypot(((x0 + x1) / 2 - W / 2) / W, ((y0 + y1) / 2 - H / 2) / H) / np.hypot(.5, .5)   # 0 가운데 ~ 1 모서리
        edge = x0 <= EDGE * W or y0 <= EDGE * H or x1 >= (1 - EDGE) * W or y1 >= (1 - EDGE) * H
        return area * (1 - 0.7 * d) * (0.3 if edge else 1.0), edge
    dishes = [(c['id'],) + info(c) for c in cands if c['dish']]
    if not dishes:
        return None
    best = max(dishes, key=lambda t: t[1])
    return [i for i, s, edge in dishes if i == best[0] or (not edge and (set_menu or s >= keep * best[1]))]


def _inside(inner, outer):
    """inner 박스 넓이 중 outer 박스와 겹치는 비율"""
    ix = max(0, min(inner[2], outer[2]) - max(inner[0], outer[0]))
    iy = max(0, min(inner[3], outer[3]) - max(inner[1], outer[1]))
    area = max((inner[2] - inner[0]) * (inner[3] - inner[1]), 1e-6)
    return ix * iy / area


def pick_in_boxes(img, cands, boxes):
    """사용자 박스 → (고른 그릇 id 목록, 마스크)
    박스 안에 찾아 둔 그릇이 있으면 그 그릇들, 없으면 그 박스 영역을 SAM으로 새로 땀"""
    W, H = img.size
    ids, extra = [], []
    for b in boxes:
        b = [max(0, min(float(b[0]), W)), max(0, min(float(b[1]), H)), max(0, min(float(b[2]), W)), max(0, min(float(b[3]), H))]
        if b[2] - b[0] < 2 or b[3] - b[1] < 2:
            continue
        hit = [c['id'] for c in cands if c['kind'] != 'other' and _inside(c['box'], b) >= BOX_INSIDE]
        if hit:
            ids += [i for i in hit if i not in ids]
        else:
            extra.append(refine(img, b))
    m = merge(img, cands, ids) if ids else None
    for r in extra:
        m = r if m is None else Image.fromarray(np.maximum(np.asarray(m), np.asarray(r)))
    return ids, m


def nukki(image, select=None, box=None, points=None, labels=None, out_dir=OUT_DIR, name=None, main_only=MAIN_ONLY,
          product_name='', boxes=None):
    """[서버용] 사진 한 장 → nukki_out (1-8과 같은 형식)

    image    : PIL 이미지, 파일 경로, 또는 업로드 bytes
    select   : 고를 그릇 id 목록 (None = main_only면 메인 그릇 위주, 아니면 기본 선택 그릇 전부)
    boxes    : 사용자가 그린 박스 목록 [[x0, y0, x1, y1], ...] (원본 픽셀 좌표). 주면 자동 선택 없이 박스 안의 그릇만
    box, points, labels: 박스 하나 + 클릭 점으로 그 그릇 하나만 SAM으로 따기 (노트북 1-7용)
    product_name: 상품명. 세트·정식·한상 등이 있으면 사진 끝에 안 걸린 그릇 전부
    name     : 저장 파일 이름 앞부분 (None이면 요청마다 새로 만듦 → 동시 요청에도 안 겹침)
    반환     : nukki_out dict (image, mask, cutout, cutout_crop 경로, size, food_box, dishes, finetuned, name)
    """
    img = _as_image(image)
    name = name or uuid.uuid4().hex[:8]
    os.makedirs(out_dir, exist_ok=True)
    if boxes:   # 사용자가 박스로 지정 → 박스 안의 그릇만 (없으면 SAM으로 새로)
        cands = candidates(img)
        select, m = pick_in_boxes(img, cands, boxes)
        if m is None or not np.asarray(m).any():
            raise NukkiNotFound('박스 안에서 음식을 찾지 못했어요. 음식이 박스 안에 들어가게 다시 그려 주세요.')
    elif box:   # 박스 하나 + 클릭 점 → 그 그릇만
        cands, select = [], []
        m = refine(img, box, points, labels)
    else:
        cands = candidates(img)
        if select is None and main_only and cands:
            select = pick_main(cands, img.size, set_menu=is_set_menu(product_name))
        m = merge(img, cands, select) if cands else None
    if m is None or not np.asarray(m).any():
        raise NukkiNotFound('음식(그릇)을 찾지 못했어요. 음식 위치를 박스로 지정해 주세요.')
    out = to_next_stage(img, m, cands, select, out_dir=out_dir, name=name)
    out['name'] = name
    return out


# 확인: 1-4에서 넣은 사진으로 한 번 실행 → 메인 그릇만 남았는지 눈으로 확인
# 박스 보정 확인: nukki(img, boxes=[[x0, y0, x1, y1]])
_t = time.time()
_chk = nukki(img)   # 세트 규칙 확인: nukki(img, product_name='봉골레 파스타 세트')
print(f"nukki() OK | {time.time() - _t:.1f}초 | 그릇 {len(_chk['dishes'])}개 {[d['label'] for d in _chk['dishes']]} | {_chk['mask']}")
plt.figure(figsize=(6, 6)); plt.imshow(Image.open(_chk['cutout'])); plt.axis('off'); plt.show()

# 2. VLM (요청 해석)

**cutout 이미지 + 사용자 입력 → 배경 생성 프롬프트 + 광고 문구**

| 단계 | 모델 | 역할 |
|---|---|---|
| 이미지·요청 이해 | Qwen2.5-VL-3B-Instruct + LoRA 파인튜닝 | 음식 사진과 요청을 함께 보고 광고 전략을 정해 JSON으로 출력 |
| 출력 정리 | 직접 작성 | JSON 파싱, 필수 문구 보정, 실패 시 기본값(fallback) 대체 |

* 입력: `nukki_out`(1단계 Segmentation 결과) + `form_data`(상품명, 가게명, 요청, 대상 고객, 분위기, 비율)
* 출력 `vlm_out`:

| 키 | 내용 | 쓰는 곳 |
|---|---|---|
| `sd_prompt` | 배경 장면 설명 (영어) | 3. 배경 생성 |
| `negative_prompt` | 생성에서 피할 요소 (영어) | 3. 배경 생성 |
| `ad_copy` | 광고 문구 (한국어) | 5. 서버 응답 (`copy`) |
| `strategy_reason` | 배경·문구 선정 이유 (한국어) | 확인용 |
| `aspect_ratio` | `1:1` / `4:5` / `9:16` | 3. 캔버스 크기 |

* 파인튜닝 데이터: 음식 사진 100장, 요청 240개 (손·인형 포함 사진, 다중 음식, 빈 입력·무의미 입력 등 예외 상황 포함)
* **음식 중복 방지**: 메인 음식은 4단계에서 원본 픽셀을 그대로 재합성하므로 `sd_prompt`에는 배경만 묘사하고, 음식이 다시 생성되지 않도록 `negative_prompt`에 `duplicate food`를 항상 포함

## 2-1. 모델 불러오기
* `Qwen2.5-VL-3B-Instruct`(fp16)에 LoRA 어댑터를 병합(`merge_and_unload`)하여 추론 속도를 베이스 모델과 동일하게 유지
* `LORA_PATH`: 파인튜닝 어댑터 위치 (없으면 베이스 모델로 동작)
* 이미 불러온 상태면 다시 불러오지 않고 건너뜀 (GPU 메모리 초과 방지)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LORA_PATH = f"{NUKKI_BASE}/qwen_vl_lora_mac"   # 파인튜닝 어댑터 경로

if 'model' not in globals() or 'processor' not in globals():
    t0 = time.time()
    print("VLM 베이스 모델 로딩 중")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VLM_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    if os.path.exists(LORA_PATH):
        print("LoRA 어댑터 적용 중")
        model = PeftModel.from_pretrained(model, LORA_PATH)
        model = model.merge_and_unload()
        vlm_mode = f"파인튜닝 ({LORA_PATH})"
    else:
        vlm_mode = "베이스 모델 (LoRA 없음)"

    model.eval()
    processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
    print(f"VLM 로딩 완료 | {vlm_mode} | {time.time() - t0:.1f}초")
else:
    print("VLM 이미 로딩됨 (건너뜀)")

## 2-2. 전처리 · 프롬프트 · 출력 정리 함수
실행만 하면 됨. 2-3 ~ 2-6에서 공통으로 쓰는 함수:

| 함수 | 입력 → 출력 |
|---|---|
| `_prep_vlm_image(nukki_out, out_dir, name)` | 투명 cutout → 회색(128) 배경 RGB 이미지 경로 |
| `_format_vlm_prompt(form_data)` | 입력 딕셔너리 → VLM 입력 텍스트 |
| `_parse_vlm_json(text)` | 모델 응답 문자열 → JSON `dict` (실패 시 `None`) |
| `_postprocess_vlm(vlm_out)` | 접지 그림자·조명 및 `duplicate food` 네거티브 문구 강제 보정 |
| `_fallback_vlm(form_data)` | 파싱·필수 키 누락 시 사용할 기본 `vlm_out` 반환 |
| `_valid_vlm(vlm_out)` | 필수 3개 키가 비어 있지 않은 문자열인지 확인 (True/False) |

* 시스템 프롬프트(`VLM_SYSTEM_PROMPT`)는 **파인튜닝 학습 데이터와 글자 하나까지 동일**해야 함
* 주요 규칙: 가게명 `미입력` 시 상호 창작 금지 / 분위기 `AI 추천` 시 자율 선택 후 `strategy_reason`에 근거 기재 / `sd_prompt`에는 배경·소품·조명만 기술

In [ ]:
import re

VLM_SYSTEM_PROMPT = """You are an expert marketing strategist specializing in helping local restaurant owners boost sales.
Analyze the provided food image (preprocessed with a neutral gray background) and the user's situational context to create an effective promotional strategy.
Generate a detailed English prompt for an AI image generator and a short, impactful Korean ad copy.

Input fields guide:
- If '가게명' (store_name) is '미입력', NEVER invent or include a store name in 'ad_copy'.
- If '대상 고객' (target_audience) is '미입력', infer the best target audience from the food image and request.
- If '원하는 분위기' (mood) is 'AI 추천', choose the most effective mood and explain it in 'strategy_reason'; otherwise, strictly reflect the requested mood in 'sd_prompt'.
- 'aspect_ratio' in your JSON output MUST match the requested '광고 비율'.

You MUST respond ONLY with a valid JSON object matching the exact structure below, without any introductory or concluding text:

{
  "sd_prompt": "English description of ONLY the background scene: surface, props, lighting, mood. The main food item already exists in the image — do NOT describe it, name it, or add another plate/serving of it. Props requested by the user (e.g. extra bread) may be included as small side items. MUST include 'soft contact shadow beneath the center object, natural directional lighting'",
"negative_prompt": "text, words, watermark, typography, floating object, bad proportions, unnatural shadows, duplicate food, extra serving of the main dish",
  "ad_copy": "Short, catchy marketing copy written in fluent Korean",
  "strategy_reason": "Strategic explanation written in Korean",
  "aspect_ratio": "Choose one from: '1:1', '4:5', or '9:16'"
}"""

SHADOW = "soft contact shadow beneath the center object"
EXTRA_NEG = "duplicate food, extra serving of the main dish"
VLM_REQUIRED_KEYS = {"sd_prompt", "ad_copy", "aspect_ratio"}


def _prep_vlm_image(nukki_out: dict, out_dir: str = OUT_DIR, name: str = "input") -> str:
    # cutout RGBA 이미지를 긴 변 512로 줄이고 투명 배경을 회색(128)으로 채운 PNG 경로 반환 (학습 데이터와 동일 형식)
    src_path = nukki_out.get("cutout_crop") or nukki_out.get("cutout") or f"{out_dir}/{name}_cutout_crop.png"
    try:
        img_vlm = Image.open(src_path)
        if img_vlm.mode in ("RGBA", "LA") or (img_vlm.mode == "P" and "transparency" in img_vlm.info):
            img_vlm = img_vlm.convert("RGBA")
            img_vlm.thumbnail((512, 512))
            gray_bg = Image.new("RGB", img_vlm.size, (128, 128, 128))
            gray_bg.paste(img_vlm, mask=img_vlm.split()[3])
            dst_path = f"{out_dir}/{name}_vlm_processed.png"
            gray_bg.save(dst_path, format="PNG")
            return dst_path
    except Exception as e:
        print(f"이미지 전처리 중 예외 발생 (원본 유지): {e}")
    return src_path


def _format_vlm_prompt(form_data: dict) -> str:
    # form_data → VLM 사용자 input 문장으로 합침
    return "\n".join([
        f"상품명: {(form_data.get('product_name') or '').strip()}",
        f"가게명: {(form_data.get('store_name') or '').strip() or '미입력'}",
        f"광고 요구사항: {(form_data.get('request_text') or '').strip()}",
        f"대상 고객: {(form_data.get('target_audience') or '').strip() or '미입력'}",
        f"원하는 분위기: {(form_data.get('mood') or '').strip() or 'AI 추천'}",
        f"광고 비율: {(form_data.get('aspect_ratio') or '').strip() or '1:1'}",
    ])


def _parse_vlm_json(text: str):
    # 모델 출력에서 JSON 블록만 추출해 파싱 (앞뒤 설명이 붙어도 허용). 실패 시 None
    s = text.strip().replace("```json", "").replace("```", "")
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b <= a:
        return None
    try:
        out = json.loads(s[a:b + 1])
        return out if isinstance(out, dict) else None
    except json.JSONDecodeError:
        return None


def _postprocess_vlm(vlm_out: dict) -> dict:
    # 생성 단계(3단계) 필수 문구 보정: 접지 그림자/자연광 및 음식 중복 방지 네거티브 추가
    sd = re.sub(r"soft drop shadow(?: under the [\w\- ]+?)?(?=,|$)", SHADOW, (vlm_out.get("sd_prompt") or "").strip())
    if SHADOW not in sd:
        sd = f"{sd.rstrip(', ')}, {SHADOW}"
    if "natural directional lighting" not in sd:
        sd = sd.replace(SHADOW, f"{SHADOW}, natural directional lighting", 1)
    vlm_out["sd_prompt"] = sd

    neg = (vlm_out.get("negative_prompt") or "text, words, watermark, typography, floating object, bad proportions, unnatural shadows").strip().rstrip(",")
    vlm_out["negative_prompt"] = neg if "duplicate food" in neg else f"{neg}, {EXTRA_NEG}"
    return vlm_out


def _fallback_vlm(form_data: dict) -> dict:
    # JSON 파싱 실패 또는 핵심 3개 키(sd_prompt, ad_copy, aspect_ratio) 누락 시 기본값
    return {
        "sd_prompt": f"a warm cozy restaurant table, {SHADOW}, natural directional lighting, photorealistic",
        "negative_prompt": f"text, words, watermark, typography, floating object, bad proportions, unnatural shadows, {EXTRA_NEG}",
        "ad_copy": "오늘의 한 끼",
        "strategy_reason": "fallback",
        "aspect_ratio": form_data.get("aspect_ratio", "1:1"),
    }

def _valid_vlm(vlm_out) -> bool:
    # 필수 3개 키가 모두 '비어 있지 않은 문자열'인지 확인
    return bool(vlm_out) and all(
        isinstance(vlm_out.get(k), str) and vlm_out[k].strip() for k in VLM_REQUIRED_KEYS
    )

print("VLM 전처리/후처리 함수 등록 완료")

## 2-3. 광고 입력값 · 이미지 준비 (노트북 테스트용)
* 오른쪽 폼에 광고 조건을 입력하고 실행 (웹 화면 입력칸과 동일 항목)
* 1-8의 `nukki_out['cutout_crop']` 투명 배경을 회색(128)으로 채워 원래 배경 간섭을 차단
* 서버(5번)로 실행할 때는 웹 요청 값이 자동으로 들어가므로 이 셀은 건너뜀

In [ ]:
#@title 광고 입력값 (오른쪽 칸에 입력하고 실행)
product_name = "두부"  #@param {type:"string"}
store_name = ""  #@param {type:"string"}
request_text = "다양한 두부의 종류에 대해서 홍보하고 싶어"  #@param {type:"string"}
target_audience = "가족"  #@param {type:"string"}
mood = "AI 추천"  #@param ["AI 추천", "깔끔한", "감성적인", "고급스러운", "활기찬"]
aspect_ratio = "4:5"  #@param ["1:1", "4:5", "9:16"]

form_data = {
    "product_name": product_name,
    "store_name": store_name,
    "request_text": request_text,
    "target_audience": target_audience,
    "mood": mood,
    "aspect_ratio": aspect_ratio,
}

processed_image_path = _prep_vlm_image(nukki_out, out_dir=OUT_DIR, name="input")
user_text = _format_vlm_prompt(form_data)

print("전처리 이미지:", processed_image_path)
print("--- VLM 입력 텍스트 ---")
print(user_text)

## 2-4. 추론
* 회색 배경 cutout(max_pixels=262144, 약 512×512) + 입력 문장 → JSON 문자열 생성
* `do_sample=False`: 동일 입력에 대해 항상 일관된 결과 출력

In [ ]:
t0 = time.time()
messages = [
    {"role": "system", "content": VLM_SYSTEM_PROMPT},
    {"role": "user", "content": [
        {"type": "image", "image": processed_image_path, "max_pixels": 262144},
        {"type": "text", "text": user_text}
    ]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to(model.device)

print("VLM 추론 중")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
print(f"추론 완료 | {time.time() - t0:.1f}초")

## 2-5. 결과 정리 → `vlm_out`
* 모델 출력에서 JSON 부분만 잘라 파싱 (앞뒤 설명 문구가 섞여도 추출 가능)
* 핵심 3개 키(`sd_prompt`, `ad_copy`, `aspect_ratio`)가 없거나 빈 값·문자열이 아니면 기본값(`fallback`)으로 대체하고, `negative_prompt` 누락 시 후처리(`_postprocess_vlm`)에서 자동 보완 → 서비스 중단 방지
* `aspect_ratio`는 사용자 지정 값으로 고정하며, 결과는 `OUT_DIR/input_vlm.json`에도 저장

In [ ]:
print("--- JSON 파싱 결과 ---")
vlm_out = _parse_vlm_json(output_text)
if vlm_out is None:
    print("JSON 파싱 실패. 원본 텍스트:\n", output_text)

if not _valid_vlm(vlm_out):
    print("fallback 사용")
    vlm_out = _fallback_vlm(form_data)

vlm_out = _postprocess_vlm(vlm_out)

if form_data.get("aspect_ratio") in ("1:1", "4:5", "9:16"):
    vlm_out["aspect_ratio"] = form_data["aspect_ratio"]

with open(f"{OUT_DIR}/input_vlm.json", "w", encoding="utf-8") as f:
    json.dump(vlm_out, f, ensure_ascii=False, indent=2)

for key, value in vlm_out.items():
    print(f"[{key}]: {value}")

## 2-6. 서버용 함수 `vlm()`
* 서버(5번)에서 요청마다 부르는 함수. `nukki_out` + `form_data` → `vlm_out` (2-5와 같은 형식)
* 2-2 셀의 함수(`_prep_vlm_image`, `_format_vlm_prompt`, `_parse_vlm_json`, `_postprocess_vlm`, `_fallback_vlm`)를 쓰므로 2-1, 2-2 셀을 먼저 실행
* 파일 이름은 `nukki_out['name']`을 따라감 → 동시 요청에도 안 겹침

| 반환 키 | 내용 |
|---|---|
| `sd_prompt` | 배경 생성용 영어 프롬프트 (접지 그림자·자연광 보정 완료) |
| `negative_prompt` | 생성 제외 요소 (`duplicate food` 포함) |
| `ad_copy` | 한국어 광고 문구 |
| `strategy_reason` | 전략 설명 (fallback 시 `'fallback'`, 모델 생략 시 미포함 가능) |
| `aspect_ratio` | 사용자 요청 비율 (`1:1` / `4:5` / `9:16`) |

In [ ]:
@torch.no_grad()
def vlm(nukki_out: dict, form_data: dict, out_dir: str = OUT_DIR) -> dict:
    """[서버용] nukki_out + form_data → vlm_out (2-5와 같은 형식)

    nukki_out : 1-9 nukki() 결과 (cutout_crop, cutout, name 사용)
    form_data : 광고 요청 딕셔너리 (product_name, store_name, request_text, target_audience, mood, aspect_ratio)
    out_dir   : 결과 JSON 저장 폴더 (기본 OUT_DIR)
    반환      : vlm_out dict (sd_prompt, negative_prompt, ad_copy, strategy_reason, aspect_ratio)
    """
    name = nukki_out.get("name", "input")
    os.makedirs(out_dir, exist_ok=True)

    # 1) cutout 이미지 회색 배경 전처리
    processed_path = _prep_vlm_image(nukki_out, out_dir=out_dir, name=name)

    # 2) 프롬프트 조립 및 추론
    user_text = _format_vlm_prompt(form_data)
    messages = [
        {"role": "system", "content": VLM_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": processed_path, "max_pixels": 262144},
            {"type": "text", "text": user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # 3) JSON 파싱, fallback, 후처리 및 JSON 저장
    vlm_out = _parse_vlm_json(output_text)
    if vlm_out is None:
        print("JSON 파싱 실패. 원본 텍스트:\n", output_text)

    if not _valid_vlm(vlm_out):
        print("fallback 사용")
        vlm_out = _fallback_vlm(form_data)

    vlm_out = _postprocess_vlm(vlm_out)

    if form_data.get("aspect_ratio") in ("1:1", "4:5", "9:16"):
        vlm_out["aspect_ratio"] = form_data["aspect_ratio"]

    with open(f"{out_dir}/{name}_vlm.json", "w", encoding="utf-8") as f:
        json.dump(vlm_out, f, ensure_ascii=False, indent=2)

    return vlm_out


# 확인: 1-8(또는 1-9) 결과와 2-3 입력값으로 한 번 실행해 서버 함수 동작 검증
if "nukki_out" in globals() and "form_data" in globals():
    _t = time.time()
    _chk_vlm = vlm(nukki_out, form_data, out_dir=OUT_DIR)
    print(f"vlm() OK | {time.time() - _t:.1f}초 | 비율={_chk_vlm['aspect_ratio']} | 문구={_chk_vlm['ad_copy']}")
else:
    print("vlm() 등록 완료 (확인 실행은 nukki_out, form_data가 있을 때)")

# 3. 배경 생성

**Segmentation된 이미지 + VLM 프롬프트 → 새 배경 사진**

| 단계 | 역할 |
|---|---|
| 캔버스 만들기 | 목표 비율 캔버스 가운데에 음식 배치 |
| 생성 마스크 | 음식은 보호, 나머지는 생성. 바닥 쪽은 그림자가 생기도록 보호를 줄임 |
| 배경 생성 | OpenAI 이미지 편집 API를 사용하여 `sd_prompt`대로 배경·테이블을 그림 |

* 입력: `nukki_out`(1단계), `vlm_out`(2단계)
* 출력: 배경이 생성된 이미지 → `gen_out` (4단계에서 원본 음식을 다시 붙임)

## 3-1. 설정
* 배경은 **OpenAI 이미지 편집 API**로 생성 (GPU 필요 없음, **호출할 때마다 요금 발생**)
* API 키: 드라이브 `AdCanvas_nukki/secrets/openai_api_key.txt`

| 설정 | 뜻 |
|---|---|
| `GEN_LONG_SIDE` | 생성 캔버스 긴 변 (1024) |
| `SUBJECT_SCALE` | 캔버스 대비 음식 크기 (0.62) |
| `PRESERVE_NATIVE_RES` | 음식을 원본 해상도로 붙임 (배경을 키움) |
| `MAX_OUTPUT_SIDE` | 최종 이미지 긴 변 상한 (4096) |

In [ ]:
import re, base64
from pathlib import Path
from PIL import ImageFilter
from IPython.display import display

GEN_MODEL = "gpt-image-2.5-sunburst"
GEN_LONG_SIDE = 1024          # 생성 캔버스 긴 변 (16의 배수)
SUBJECT_SCALE = 0.62          # 캔버스 대비 음식 크기 비율
PRESERVE_NATIVE_RES = True    # True: 배경을 업스케일하고 음식은 원본 해상도로 합성
MAX_OUTPUT_SIDE = 4096        # 최종 출력 긴 변 상한

# API 키 로드 (없으면 즉시 중단)
API_KEY_PATH = Path(NUKKI_BASE) / 'secrets' / 'openai_api_key.txt'
if API_KEY_PATH.exists():
    os.environ['OPENAI_API_KEY'] = API_KEY_PATH.read_text().strip()
    print('OpenAI API 키: 드라이브 파일에서 로드')
elif os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키: 환경변수 사용')
else:
    raise FileNotFoundError(f'{API_KEY_PATH} 에 API 키 파일이 없습니다.')

## 3-2. 배경 생성 · 재합성 함수

| 함수 | 역할 |
|---|---|
| `build_canvas()` | 음식을 캔버스 가운데에 배치 + 재합성용 원본 준비 (`paste_alpha`) |
| `generation_mask_inline()` | 생성 마스크 (흰색 = 생성, 검은색 = 음식 보호) |
| `edit_via_openai()` | OpenAI 배경 생성. `input_fidelity="high"`로 음식을 최대한 그대로 유지 (지원 안 하면 빼고 다시 요청) |
| `paste_back()` | 생성 결과 위에 원본 음식을 다시 붙임 (4단계) |
| `evaluate()` | 음식이 원본 그대로인지 평가 (4단계) |

**재합성 보완 (실험하면서 고친 문제)**

| 문제 | 해결 | 설정 |
|---|---|---|
| 모델이 음식까지 다시 그려서 크기·위치가 달라짐 → 원본과 겹쳐 보임 | 생성 결과에서 음식 위치·크기를 찾아(특징점 매칭) 그 자리에 붙임 | `ALIGN_CFG` |
| 사진 끝에 잘린 반찬 그릇을 붙이면 반쪽 그릇이 보임 | 잘린 반찬 그릇은 붙이지 않고 모델이 그린 온전한 그릇을 씀 | `CUT_BAND`, `DROP_CUT_SIDE` |
| 메인 그릇이 잘렸을 때 직선 경계가 보임 | 잘린 쪽으로 갈수록 원본을 서서히 투명하게 → 모델이 이어 그린 부분과 연결 | `CUT_FADE` |
| 누끼 외곽이 딱딱함 | 외곽을 1px 정도 부드럽게 | `EDGE_FEATHER` |

* OpenAI 마스크 규칙: 알파 0(투명) = 생성, 알파 255 = 보존

In [ ]:
import cv2
# ---------- 마스크 연산 ----------
def binarize(m: Image.Image, threshold: int = 127) -> Image.Image:
    a = np.asarray(m.convert("L"), np.uint8)
    return Image.fromarray(np.where(a > threshold, 255, 0).astype(np.uint8), "L")

def _morph(m: Image.Image, radius: int, flt) -> Image.Image:
    # 큰 커널 대신 최대 3px씩 나눠 적용 (정확히 radius만큼)
    out, done = m.convert("L"), 0
    while done < radius:
        step = min(3, radius - done)
        out = out.filter(flt(2 * step + 1))
        done += step
    return out

def dilate(m, radius): return _morph(m, radius, ImageFilter.MaxFilter)
def erode(m, radius):  return _morph(m, radius, ImageFilter.MinFilter)

def feather(m: Image.Image, radius: int) -> Image.Image:
    return m.convert("L") if radius <= 0 else m.convert("L").filter(ImageFilter.GaussianBlur(radius))

def generation_mask_inline(food_mask, protect_px=3, blend_px=6, floor_ratio=0.20, feather_px=4):
    # 반환값: 255 = 배경 생성 영역, 0 = 음식 보호 영역
    b = binarize(food_mask)
    a = np.asarray(b, np.uint8) > 127
    if not a.any():
        raise ValueError("음식 마스크가 비어 있습니다 (알파 채널 확인 필요).")
    protected = np.asarray(dilate(b, protect_px), np.float32)
    shrunk = np.asarray(erode(b, blend_px), np.float32)

    # 바닥면(아래쪽 floor_ratio 구간)은 보호 영역을 줄여 접지 그림자가 자연스럽게 생성되도록 함
    ys, _ = np.where(a)
    top, bottom = ys.min(), ys.max()
    split = bottom - (bottom - top) * float(np.clip(floor_ratio, 0, 1))
    band = max(8.0, feather_px * 3.0)
    rows = np.arange(b.height, dtype=np.float32)
    w = np.clip((rows - (split - band / 2)) / band, 0, 1)[:, None]

    blended = np.clip(protected * (1 - w) + shrunk * w, 0, 255).astype(np.uint8)
    return feather(Image.fromarray(255 - blended, "L"), feather_px)

def to_openai_mask(image: Image.Image, gen_mask: Image.Image) -> Image.Image:
    # OpenAI 규칙: 알파=0 → 편집(생성), 알파=255 → 보존
    m = image.convert("RGBA")
    m.putalpha(ImageOps.invert(binarize(gen_mask)))
    return m

# ---------- 비율 / 캔버스 ----------
def parse_ratio(v):
    if isinstance(v, (list, tuple)) and len(v) == 2:
        rw, rh = float(v[0]), float(v[1])
    elif isinstance(v, str):
        parts = [p for p in re.split(r"\s*[:x/×,]\s*", v.strip()) if p]
        if len(parts) != 2:
            raise ValueError(f"비율 형식을 해석할 수 없습니다: {v!r}")
        rw, rh = float(parts[0]), float(parts[1])
    elif isinstance(v, (int, float)):
        rw, rh = float(v), 1.0
    else:
        raise ValueError(f"비율 형식을 해석할 수 없습니다: {v!r}")
    if rw <= 0 or rh <= 0:
        raise ValueError(f"비율은 양수여야 합니다: {v!r}")
    return rw, rh

def _snap16(v): return max(16, int(round(v / 16)) * 16)

def canvas_size_for_ratio(ratio, long_side=1024):
    rw, rh = ratio
    if rw >= rh:
        return _snap16(long_side), _snap16(long_side * rh / rw)
    return _snap16(long_side * rw / rh), _snap16(long_side)

# 재합성 경계 처리
CUT_BAND = 0.015      # 사진 테두리에서 이 비율(짧은 변 기준) 안까지 마스크가 오면 '사진에서 잘린 그릇'으로 봄 (누끼는 보통 테두리 몇 px 앞에서 끝남)
DROP_CUT_SIDE = True  # 잘린 그릇 중 메인이 아닌 것(반찬 등)은 원본을 붙이지 않고 모델이 그린 것을 그대로 씀
CUT_FADE = 0.12       # 잘린 메인 그릇: 그릇 크기의 12% 구간에서 원본을 서서히 투명하게 → 모델이 이어 그린 부분으로 연결
EDGE_FEATHER = 1.0    # 누끼 외곽을 캔버스 기준 이 px만큼 부드럽게 (원본 해상도에서는 배율만큼 커짐)


def paste_alpha(alpha: Image.Image, scale: float, cut_band=CUT_BAND, drop_cut_side=DROP_CUT_SIDE,
                cut_fade=CUT_FADE, edge_feather=EDGE_FEATHER):
    """재합성에 쓸 알파 (원본 해상도) + 정보.
    1) 마스크를 그릇 단위(연결된 덩어리)로 나누고, 사진 테두리에 닿은(잘린) 덩어리를 찾음
       - 메인(가장 큰 덩어리)이 아니면: 붙이지 않음 → 모델이 온전한 모양으로 다시 그린 것 사용
       - 메인이면: 테두리 쪽으로 갈수록 투명하게 → 모델이 이어 그린 부분과 자연스럽게 연결
    2) 외곽을 살짝 안쪽으로 줄인 뒤 흐리게 → 원래 배경색 테두리(halo) 없이 부드러운 경계"""
    a = np.asarray(alpha.convert("L")) > 127
    info = {"cut_main": False, "dropped_parts": 0}
    if not a.any():
        return alpha.convert("L"), info
    H, W = a.shape
    out = a.astype(np.float32)

    # 1) 그릇 단위로 나눠 잘린 덩어리 찾기
    n, lab, stats, _ = cv2.connectedComponentsWithStats(a.astype(np.uint8), connectivity=8)
    main = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    b = max(3, int(round(cut_band * min(W, H))))
    border = np.zeros_like(a)
    border[:b, :] = border[-b:, :] = True
    border[:, :b] = border[:, -b:] = True
    cut_labels = set(np.unique(lab[border & a]).tolist()) - {0}
    for k in cut_labels:
        if k != main and drop_cut_side:
            out[lab == k] = 0
            info["dropped_parts"] += 1
    if main in cut_labels and cut_fade > 0:
        comp = lab == main
        seed = border & comp
        x, y, w, h = stats[main, :4]
        fade_px = max(4.0, cut_fade * max(w, h))
        dist = cv2.distanceTransform((~seed).astype(np.uint8), cv2.DIST_L2, 5)
        t = np.clip(dist / fade_px, 0, 1)
        out[comp] *= (t * t * (3 - 2 * t))[comp]      # smoothstep
        info["cut_main"] = True

    # 2) 외곽 부드럽게
    r = edge_feather / max(scale, 1e-6)
    if r >= 0.5:
        k = max(1, int(round(r)))
        er = cv2.erode(a.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1)))
        soft = cv2.GaussianBlur(er.astype(np.float32), (0, 0), sigmaX=r)
        out = np.minimum(out, soft)
    return Image.fromarray(np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8), "L"), info


def build_canvas(src_rgba: Image.Image, cw: int, ch: int, subject_scale: float) -> dict:
    alpha = src_rgba.getchannel("A")
    bbox = alpha.point(lambda v: 255 if v > 0 else 0).getbbox()
    if bbox is None:
        raise ValueError("알파 채널이 비어 있습니다.")
    sw, sh = bbox[2] - bbox[0], bbox[3] - bbox[1]
    scale = subject_scale * min(cw / sw, ch / sh)
    pw, ph = max(1, round(sw * scale)), max(1, round(sh * scale))
    cx, cy = (cw - pw) // 2, (ch - ph) // 2

    # 재합성용 원본: 잘린 부분 페이드 + 부드러운 외곽 (페이드 계산은 사진 테두리 기준이라 자르기 전에)
    src = src_rgba.copy()
    pa, cut_info = paste_alpha(alpha, scale)
    src.putalpha(pa)
    src = src.crop(bbox)

    # 캔버스(모델 입력)에는 원래 마스크 그대로 → 모델이 잘린 접시 모양을 보고 이어 그림
    full = src_rgba.crop(bbox).resize((pw, ph), Image.LANCZOS)
    canvas = Image.new("RGB", (cw, ch), (127, 127, 127))
    canvas.paste(full.convert("RGB"), (cx, cy), full)

    # 보호 영역 = 원본이 완전히 덮는 곳만 (페이드 구간은 모델이 새로 그림)
    fg = src.resize((pw, ph), Image.LANCZOS)
    food_mask = Image.new("L", (cw, ch), 0)
    food_mask.paste(fg.getchannel("A").point(lambda v: 255 if v >= 250 else 0), (cx, cy))
    return {"src": src, "fg": fg, "canvas": canvas, "food_mask": food_mask,
            "box": (cx, cy, cx + pw, cy + ph), "scale": scale, "cut": cut_info}

# ---------- OpenAI 호출 ----------
def edit_via_openai(image: Image.Image, gen_mask: Image.Image, prompt_text: str, model: str = None) -> Image.Image:
    from openai import OpenAI
    client = OpenAI()

    img_buf = io.BytesIO(); image.convert("RGBA").save(img_buf, format="PNG"); img_buf.seek(0)
    mask_buf = io.BytesIO(); to_openai_mask(image, gen_mask).save(mask_buf, format="PNG"); mask_buf.seek(0)
    img_buf.name, mask_buf.name = "image.png", "mask.png"

    kw = dict(model=model or GEN_MODEL, prompt=prompt_text, image=img_buf, mask=mask_buf,
              size=f"{image.width}x{image.height}", quality="high")
    try:
        # input_fidelity="high": 입력 이미지(음식)를 최대한 그대로 유지 → 음식을 다시 그려 겹치는 문제 감소
        response = client.images.edit(**kw, input_fidelity="high")
    except Exception as e:
        if "input_fidelity" not in str(e):
            raise
        print("  안내: 이 모델은 input_fidelity를 지원하지 않아 빼고 다시 요청")
        img_buf.seek(0); mask_buf.seek(0)
        response = client.images.edit(**kw)
    out = Image.open(io.BytesIO(base64.b64decode(response.data[0].b64_json))).convert("RGB")
    if out.size != image.size:
        print(f"  안내: 반환 크기 {out.size} → 캔버스 크기 {image.size}로 맞춤")
        out = out.resize(image.size, Image.LANCZOS)
    return out

# ---------- 재합성 (4단계에서 사용) ----------
# 주의: OpenAI 편집 모델은 마스크를 '참고'만 하고 장면 전체를 다시 그림 → 음식이 캔버스보다 커지거나 밀린 채로 나올 수 있음
#       그래서 생성 결과에서 음식이 실제로 그려진 위치·크기를 특징점 매칭으로 찾아서, 원본을 그 자리·크기에 맞춰 붙임
ALIGN_CFG = dict(
    min_inliers=12,       # 매칭 성공으로 볼 최소 일치점 수
    scale_range=(0.6, 1.8),  # 이 범위를 벗어난 크기 변화는 잘못된 매칭으로 보고 무시
    max_rot_deg=4.0,      # 회전이 이보다 크면 잘못된 매칭으로 보고 무시
)


def _feature_detector():
    if hasattr(cv2, "SIFT_create"):
        return cv2.SIFT_create(nfeatures=4000), cv2.NORM_L2
    return cv2.ORB_create(nfeatures=4000), cv2.NORM_HAMMING


def estimate_food_transform(generated: Image.Image, layout: dict) -> dict:
    """캔버스의 음식 → 생성 결과의 음식 사이 변환 (크기 s, 이동 tx, ty) 추정
    gen 좌표 = s * canvas 좌표 + (tx, ty). 실패하면 ok=False (캔버스 위치 그대로 사용)"""
    ident = {"ok": False, "s": 1.0, "tx": 0.0, "ty": 0.0, "inliers": 0}
    can = cv2.cvtColor(np.asarray(layout["canvas"].convert("RGB")), cv2.COLOR_RGB2GRAY)
    gen = cv2.cvtColor(np.asarray(generated.convert("RGB")), cv2.COLOR_RGB2GRAY)
    if gen.shape != can.shape:
        return ident
    food = np.asarray(erode(layout["food_mask"], 4)) > 127      # 음식 안쪽에서만 특징점
    det, norm = _feature_detector()
    k1, d1 = det.detectAndCompute(can, food.astype(np.uint8) * 255)
    k2, d2 = det.detectAndCompute(gen, None)
    if d1 is None or d2 is None or len(k1) < ALIGN_CFG["min_inliers"] or len(k2) < ALIGN_CFG["min_inliers"]:
        return ident
    pairs = cv2.BFMatcher(norm).knnMatch(d1, d2, k=2)
    good = [p[0] for p in pairs if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < ALIGN_CFG["min_inliers"]:
        return ident
    src = np.float32([k1[m.queryIdx].pt for m in good])
    dst = np.float32([k2[m.trainIdx].pt for m in good])
    M, inl = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0,
                                         maxIters=5000, confidence=0.995)
    if M is None or inl is None or int(inl.sum()) < ALIGN_CFG["min_inliers"]:
        return ident
    a, b = float(M[0, 0]), float(M[1, 0])
    s = float(np.hypot(a, b))
    rot = abs(np.degrees(np.arctan2(b, a)))
    lo, hi = ALIGN_CFG["scale_range"]
    if not (lo <= s <= hi) or rot > ALIGN_CFG["max_rot_deg"]:
        return dict(ident, inliers=int(inl.sum()), rejected=f"s={s:.3f}, rot={rot:.1f}")
    # 회전은 무시하고 크기+이동만 다시 맞춤 (일치점 기준 최소제곱)
    m = inl.ravel().astype(bool)
    p, q = src[m], dst[m]
    pc, qc = p.mean(0), q.mean(0)
    s = float(((p - pc) * (q - qc)).sum() / max(((p - pc) ** 2).sum(), 1e-6))
    tx, ty = (qc - s * pc).tolist()
    return {"ok": True, "s": round(s, 4), "tx": round(tx, 2), "ty": round(ty, 2), "inliers": int(m.sum())}


def paste_back(generated: Image.Image, layout: dict, preserve_native=True, max_side=4096, align=True):
    """생성 결과 위에 원본 음식 픽셀을 다시 덮음.
    align=True면 생성 결과에서 음식이 실제로 그려진 위치·크기에 맞춰 붙임 (layout['align']에 결과 기록)"""
    cw, ch = generated.size
    t = estimate_food_transform(generated, layout) if align else {"ok": False, "s": 1.0, "tx": 0.0, "ty": 0.0}
    layout["align"] = t
    if align and not t["ok"]:
        print(f"  안내: 생성 결과에서 음식 위치를 찾지 못해 캔버스 위치 그대로 붙임 ({t})")

    # 생성 결과 기준 음식 박스 (캔버스 좌표 → 생성 결과 좌표)
    s = t["s"]
    x0, y0, x1, y1 = layout["box"]
    gx, gy = s * x0 + t["tx"], s * y0 + t["ty"]
    gw, gh = s * (x1 - x0), s * (y1 - y0)
    eff_scale = layout["scale"] * s          # 원본 음식 → 생성 결과 배율

    up = 1.0
    if preserve_native:
        up = max(1.0, 1.0 / eff_scale)
        if max(cw, ch) * up > max_side:
            up = max(1.0, max_side / max(cw, ch))
    W, H = round(cw * up), round(ch * up)
    bg = generated.resize((W, H), Image.LANCZOS) if (W, H) != (cw, ch) else generated.copy()

    pos = (round(gx * up), round(gy * up))
    src = layout["src"]
    fw, fh = max(1, round(gw * up)), max(1, round(gh * up))
    if abs(fw - src.width) <= 1 and abs(fh - src.height) <= 1:
        fg = src                                   # 원본 픽셀 그대로
    else:
        fg = src.resize((fw, fh), Image.LANCZOS)
    bg.paste(fg.convert("RGB"), pos, fg.getchannel("A"))   # 캔버스 밖으로 나간 부분은 PIL이 잘라냄
    return bg, fg, pos

# ---------- 평가 (4단계에서 사용) ----------
def evaluate(final: Image.Image, fg: Image.Image, pos, src: Image.Image) -> dict:
    x, y = pos
    region = np.asarray(final.crop((x, y, x + fg.width, y + fg.height)).convert("RGB"), np.float32)
    ref = np.asarray(fg.convert("RGB"), np.float32)
    core = np.asarray(fg.getchannel("A")) == 255   # 완전 불투명 음식 픽셀
    # 음식이 결과 이미지 밖으로 나간 부분은 평가에서 제외
    yy, xx = np.indices(core.shape)
    core &= (xx + x >= 0) & (xx + x < final.width) & (yy + y >= 0) & (yy + y < final.height)
    diff = np.abs(region - ref)[core]
    mae = float(diff.mean()) if diff.size else float("nan")
    return {
        "food_mae": round(mae, 4),
        "food_max_abs_err": float(diff.max()) if diff.size else float("nan"),
        "food_pixels_checked": int(core.sum()),
        "food_native_resolution": fg.size == src.size,
        "food_size": list(fg.size),
        "food_pos": [int(x), int(y)],
        "output_size": list(final.size),
        "food_preserved": bool(diff.size and mae < 0.5),
    }

def review_sheet(canvas, gen_mask, generated, final, height=512):
    tiles = [canvas, Image.merge("RGB", [gen_mask] * 3), generated, final.convert("RGB")]
    tiles = [t.resize((round(t.width * height / t.height), height), Image.LANCZOS) for t in tiles]
    sheet = Image.new("RGB", (sum(t.width for t in tiles) + 8 * (len(tiles) - 1), height), (255, 255, 255))
    x = 0
    for t in tiles:
        sheet.paste(t, (x, 0)); x += t.width + 8
    return sheet

print("배경 생성 함수 등록 완료")

## 3-3. 입력 준비 → 캔버스 · 생성 마스크 미리보기
* 음식: `nukki_out`의 원본 + 마스크로 cutout(RGBA)을 만듦
* 비율: `vlm_out['aspect_ratio']` (`1:1`, `4:5`, `9:16`)
* 프롬프트: `sd_prompt` + 음식 보존 문장 + `negative_prompt`(OpenAI에는 negative 입력이 없어서 `Avoid: ...` 문장으로 붙임)
* 미리보기: **캔버스 | 생성 마스크(흰색 = 생성) | OpenAI 전송 마스크(체크무늬 = 생성)**

In [ ]:
PRESERVE_TEXT = ("The food and its dish in the center are the hero subject and must remain exactly unchanged; "
                 "only generate the surrounding background and tabletop.")


def build_gen_prompt(vlm: dict) -> str:
    """vlm_out → OpenAI 편집 프롬프트 (sd_prompt + 음식 보존 지시 + negative를 Avoid 문장으로)"""
    parts = [vlm.get('sd_prompt', '').strip().rstrip('.') + '.', PRESERVE_TEXT]
    neg = (vlm.get('negative_prompt') or '').strip().rstrip('.')
    if neg:
        parts.append(f"Avoid: {neg}.")
    return ' '.join(parts)


def load_food_rgba(nukki: dict) -> Image.Image:
    """nukki_out의 원본 + 마스크 → RGBA cutout (흰색 마스크 = 불투명)"""
    rgba = Image.open(nukki['image']).convert('RGB')
    rgba.putalpha(Image.open(nukki['mask']).convert('L').resize(rgba.size, Image.NEAREST))
    return rgba


# 미리보기: 1-8 nukki_out, 2-5 vlm_out이 있을 때만 (서버만 켤 때는 건너뜀)
if 'nukki_out' in globals() and 'vlm_out' in globals():
    try:
        GEN_RATIO = parse_ratio(vlm_out.get('aspect_ratio') or '1:1')
    except ValueError as e:
        print(f"비율 오류: {e} → 1:1 사용")
        GEN_RATIO = (1.0, 1.0)
    GEN_PROMPT = build_gen_prompt(vlm_out)
    CW, CH = canvas_size_for_ratio(GEN_RATIO, GEN_LONG_SIDE)

    food_rgba = load_food_rgba(nukki_out)
    layout = build_canvas(food_rgba, CW, CH, SUBJECT_SCALE)
    gen_mask = generation_mask_inline(layout['food_mask'], protect_px=3, blend_px=6, feather_px=4)

    print(f"비율: {vlm_out.get('aspect_ratio')} → 캔버스 {CW}x{CH} | 음식 축소 비율 {layout['scale']:.3f}")
    print(f"프롬프트: {GEN_PROMPT}")

    api_mask = to_openai_mask(layout['canvas'], gen_mask)
    checker = Image.fromarray(((np.indices((CH, CW)).sum(0) // 16 % 2) * 60 + 160).astype(np.uint8)).convert('RGB')
    checker.paste(api_mask.convert('RGB'), (0, 0), api_mask.getchannel('A'))
    display(Image.fromarray(np.hstack([np.array(layout['canvas']),
                                       np.array(Image.merge('RGB', [gen_mask] * 3)),
                                       np.array(checker)])))
else:
    print('미리보기 건너뜀 (nukki_out / vlm_out 없음). 서버용 함수는 등록됨')

## 3-4. 배경 생성 (OpenAI API 호출) → `gen_out`
* **실행할 때마다 요금 발생** → 3-3 미리보기를 확인하고 실행
* 결과는 `OUT_DIR`에 저장하고 `gen_out`으로 4단계에 넘김


In [ ]:
t0 = time.time()
print("OpenAI 배경 생성 요청 중...")
generated_raw = edit_via_openai(layout['canvas'], gen_mask, GEN_PROMPT)
print(f"생성 완료 | {time.time() - t0:.1f}초")

gen_paths = {k: f'{OUT_DIR}/input_{k}.png' for k in ('canvas', 'gen_mask', 'generated')}
layout['canvas'].save(gen_paths['canvas'])
gen_mask.save(gen_paths['gen_mask'])
generated_raw.save(gen_paths['generated'])

gen_out = dict(gen_paths, ratio=vlm_out.get('aspect_ratio'), canvas_size=[CW, CH],
               food_box=list(layout['box']), scale=layout['scale'], prompt=GEN_PROMPT, model=GEN_MODEL)
json.dump(gen_out, open(f'{OUT_DIR}/input_gen.json', 'w'), ensure_ascii=False, indent=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
ax[0].imshow(layout['canvas']); ax[0].set_title('canvas')
ax[1].imshow(generated_raw); ax[1].set_title('generated (model output)')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()

# 4. 원본 음식 재합성

**생성된 배경 + 원본 음식 → 최종 광고 이미지**

* 모델이 음식을 조금 바꿔 그려도, 원본 음식 픽셀을 다시 덮어서 음식은 항상 실제 사진 그대로
* 출력: 최종 광고 이미지 → `final_out`

## 4-1. 원본 음식 재합성 + 평가

1. **위치 맞추기**: 생성 결과에서 음식이 그려진 위치·크기를 찾음 (못 찾으면 캔버스 위치 그대로)
2. **배경 키우기**: 음식이 원본 해상도가 되도록 배경을 키움 (긴 변 `MAX_OUTPUT_SIDE` 이하)
3. **원본 붙이기**: 잘린 반찬 그릇은 빼고, 잘린 메인 그릇은 서서히 투명하게, 외곽은 부드럽게 붙임

평가 (`metrics`)

| 키 | 뜻 |
|---|---|
| `food_mae` | 음식 픽셀이 원본과 얼마나 다른지 (0 = 완전히 같음, 0.5 미만 = 통과) |
| `food_native_resolution` | 음식을 원본 해상도로 붙였는지 |
| `align` | 위치 보정 결과 (`s` = 크기 배율, `tx`·`ty` = 이동, `ok` = 찾았는지) |

* 비교 이미지: **캔버스 | 생성 마스크 | 모델 출력 | 최종 결과**

In [ ]:
final_img, fg_used, fg_pos = paste_back(generated_raw, layout,
                                        preserve_native=PRESERVE_NATIVE_RES, max_side=MAX_OUTPUT_SIDE)
metrics = evaluate(final_img, fg_used, fg_pos, layout['src'])
metrics.update(model=GEN_MODEL, ratio=vlm_out.get('aspect_ratio'), canvas_size=[CW, CH], prompt=GEN_PROMPT,
               align=layout.get('align'))
print(f"음식 위치 보정: {layout.get('align')}")

status = '통과' if metrics['food_preserved'] else '확인 필요'
print(f"food_mae={metrics['food_mae']} / 원본 해상도={metrics['food_native_resolution']} "
      f"/ 출력 {metrics['output_size'][0]}x{metrics['output_size'][1]} → {status}")
review = review_sheet(layout['canvas'], gen_mask, generated_raw, final_img)
display(review)   # 캔버스 | 생성 마스크 | 모델 원본 출력 | 최종 결과

## 4-2. 최종 결과 저장
* 최종 이미지, 비교 이미지, 평가 결과를 `OUT_DIR`에 저장하고 `final_out`으로 5단계(서버)에 넘김

In [ ]:
final_paths = {'final': f'{OUT_DIR}/input_final.png', 'review': f'{OUT_DIR}/input_review.png'}
final_img.save(final_paths['final'])
review.save(final_paths['review'])
json.dump(metrics, open(f'{OUT_DIR}/input_metrics.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

final_out = dict(final_paths, metrics=metrics, ad_copy=vlm_out.get('ad_copy', ''),
                 aspect_ratio=vlm_out.get('aspect_ratio'))
json.dump(final_out, open(f'{OUT_DIR}/input_final.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

display(Image.open(final_paths['final'])) # 최종 결과 (input_final.png)
print('저장:', final_paths['final'])
print('광고 문구:', final_out['ad_copy'])

## 4-3. 서버용 함수 `generate()`
* 서버(5번)에서 요청마다 부르는 함수. `nukki_out` + `vlm_out` → `final_out` (4-2와 같은 형식)
* 3-3 ~ 4-2를 한 번에: 캔버스 구성 → 생성 마스크 → OpenAI 배경 생성 → 원본 음식 재합성 → 저장
* 3-2, 3-3 셀의 함수(`build_canvas`, `build_gen_prompt`, `load_food_rgba` 등)를 쓰므로 그 셀들을 먼저 실행
* 파일 이름은 `nukki_out['name']`을 따라감 → 동시 요청에도 안 겹침
* **호출할 때마다 API 요금 발생**. 아래 확인 실행은 `CHECK_GENERATE = True`일 때만 함

| 반환 키 | 내용 |
|---|---|
| `final` | 최종 광고 이미지 경로 (서버가 `imageUrl`로 바꿔 보냄) |
| `generated` | 모델 원본 출력 경로 (재합성 전) |
| `ad_copy` | VLM 광고 문구 (서버 응답의 `copy`) |
| `aspect_ratio`, `metrics`, `name` | 비율, 음식 보존 평가, 요청 이름 |

In [ ]:
def generate(nukki_out, vlm_out, out_dir=OUT_DIR, name=None, save_review=False):
    """[서버용] nukki_out + vlm_out → final_out (4-2와 같은 형식)

    nukki_out  : 1-9 nukki() 결과 (image, mask 경로 사용)
    vlm_out    : VLM 결과 (sd_prompt, negative_prompt, ad_copy, aspect_ratio)
    name       : 저장 파일 이름 앞부분 (None이면 nukki_out['name'], 그것도 없으면 새로 만듦)
    save_review: True면 비교 이미지(캔버스 | 생성 마스크 | 모델 출력 | 최종)도 저장
    반환       : final_out dict (final, generated, ad_copy, aspect_ratio, metrics, name)
    """
    name = name or nukki_out.get('name') or uuid.uuid4().hex[:8]
    os.makedirs(out_dir, exist_ok=True)
    ratio_str = vlm_out.get('aspect_ratio') or '1:1'
    try:
        ratio = parse_ratio(ratio_str)
    except ValueError:
        ratio_str, ratio = '1:1', (1.0, 1.0)
    cw, ch = canvas_size_for_ratio(ratio, GEN_LONG_SIDE)

    # ③ 배경 생성
    prompt = build_gen_prompt(vlm_out)
    layout = build_canvas(load_food_rgba(nukki_out), cw, ch, SUBJECT_SCALE)
    gen_mask = generation_mask_inline(layout['food_mask'], protect_px=3, blend_px=6, feather_px=4)
    t0 = time.time()
    generated = edit_via_openai(layout['canvas'], gen_mask, prompt)
    api_sec = time.time() - t0

    # ④ 원본 음식 재합성 + 평가
    final_img, fg, pos = paste_back(generated, layout, preserve_native=PRESERVE_NATIVE_RES, max_side=MAX_OUTPUT_SIDE)
    metrics = evaluate(final_img, fg, pos, layout['src'])
    metrics.update(model=GEN_MODEL, ratio=ratio_str, canvas_size=[cw, ch], prompt=prompt, api_sec=round(api_sec, 1),
                   align=layout.get('align'))

    # 저장
    paths = {k: f'{out_dir}/{name}_{k}.png' for k in ('generated', 'final')}
    generated.save(paths['generated'])
    final_img.save(paths['final'])
    if save_review:
        paths['review'] = f'{out_dir}/{name}_review.png'
        review_sheet(layout['canvas'], gen_mask, generated, final_img).save(paths['review'])
    final_out = dict(paths, name=name, ad_copy=vlm_out.get('ad_copy', ''), aspect_ratio=ratio_str, metrics=metrics)
    json.dump(final_out, open(f'{out_dir}/{name}_final.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    return final_out


# 확인: 1~2단계 결과로 한 번 실행 (API 요금 발생 → 필요할 때만 True)
CHECK_GENERATE = False
if CHECK_GENERATE and 'nukki_out' in globals() and 'vlm_out' in globals():
    _t = time.time()
    _chk = generate(nukki_out, vlm_out, name='check')
    print(f"generate() OK | {time.time() - _t:.1f}초 | food_mae={_chk['metrics']['food_mae']} | {_chk['final']}")
    display(Image.open(_chk['final']))
else:
    print('generate() 등록 완료 (확인 실행은 CHECK_GENERATE = True + 1-8, 2-5 결과가 있을 때)')

# 5. 서버 (FastAPI + ngrok)
* 웹 화면 `POST /generate` → ① nukki() → ② vlm() → ③④ generate() → `{imageUrl, copy}` 응답

| 단계 | 함수 | 입력 → 출력 |
|---|---|---|
| Segmentation | `nukki(image, product_name=...)` | 업로드 사진 → `nukki_out` (1-9) |
| VLM | `vlm(nukki_out, form_data)` | → `vlm_out` |
| 생성·재합성 | `generate(nukki_out, vlm_out)` | → `final_out` (최종 이미지, 광고 문구) |

* 응답 코드

| 코드 | 뜻 |
|---|---|
| 200 | 성공 `{imageUrl, copy: {headline, body, hashtags}}` |
| 400 | boxes 형식 오류 |
| 422 | 그릇을 찾지 못함 (사용자 박스로 다시 요청) |
| 500 | 서버 오류 |
| 501 | 정의되지 않은 단계 함수 (`/health`에서 확인) |

* ngrok 토큰은 코랩 보안 비밀 `NGROK_AUTHTOKEN`에 저장

## 5-1. 서버 앱

In [ ]:
import base64, io, json, re, threading, time, traceback

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

# 웹 화면 입력값 → VLM form_data (2-3 형식)
MOOD_KO = {'auto': 'AI 추천', 'clean': '깔끔한', 'emotional': '감성적인', 'luxury': '고급스러운', 'lively': '활기찬'}
VLM_RATIOS = ('1:1', '4:5', '9:16')


def to_form_data(product_name, store_name, prompt, target, mood, ratio):
    return {'product_name': product_name, 'store_name': store_name or '', 'request_text': prompt,
            'target_audience': target or '', 'mood': MOOD_KO.get(mood, mood or 'AI 추천'),
            'aspect_ratio': ratio if ratio in VLM_RATIOS else '1:1'}


def to_copy(ad_copy, product_name=''):
    """VLM ad_copy(문장) → 웹 화면 copy {headline, body, hashtags}"""
    text = (ad_copy or '').strip()
    tags = re.findall(r'#\S+', text)
    lines = [l.strip() for l in re.sub(r'#\S+', '', text).splitlines() if l.strip()]
    if len(lines) <= 1:   # 한 줄이면 첫 문장을 제목으로
        parts = re.split(r'(?<=[.!?…])\s+', lines[0]) if lines else []
        lines = [parts[0], ' '.join(parts[1:])] if len(parts) > 1 else (parts or [product_name])
    return {'headline': lines[0], 'body': '\n'.join(lines[1:]), 'hashtags': tags}


def to_data_url(path, max_side=2048, quality=90):
    """이미지 파일 → JPEG data URL (응답에 바로 담아 보냄)"""
    im = Image.open(path).convert('RGB')
    im.thumbnail((max_side, max_side), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=quality)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()


def parse_boxes(text):
    """boxes(JSON 문자열) → [[x0, y0, x1, y1], ...] / 비어 있으면 None"""
    if not text:
        return None
    try:
        boxes = [[float(v) for v in b] for b in json.loads(text) if len(b) == 4]
    except (ValueError, TypeError):
        raise ValueError('boxes 형식이 맞지 않아요. [[x0, y0, x1, y1], ...] 형식이어야 해요.')
    return [[min(b[0], b[2]), min(b[1], b[3]), max(b[0], b[2]), max(b[1], b[3])] for b in boxes] or None


def _stage(name):
    fn = globals().get(name)
    if fn is None:
        raise NotImplementedError(f'{name}() 함수가 정의되지 않았어요')
    return fn


app = FastAPI(title='AdCanvas')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
gpu_lock = threading.Lock()   # 요청이 동시에 와도 GPU는 한 번에 하나만


@app.get('/health')
def health():
    return {'ok': True, 'stages': {n: n in globals() for n in ('nukki', 'vlm', 'generate')}}


@app.post('/generate')
def generate_api(image: UploadFile = File(...), product_name: str = Form(...), prompt: str = Form(...),
                 store_name: str = Form(''), target: str = Form(''), mood: str = Form('auto'),
                 ratio: str = Form('1:1'), boxes: str = Form(''), backend: str = Form('')):
    """사진 + 입력값 (+ 사용자 박스) → {imageUrl, copy}. backend는 웹 화면 호환용으로만 받음"""
    t0 = time.time()
    try:
        user_boxes = parse_boxes(boxes)
    except ValueError as e:
        return JSONResponse(status_code=400, content={'message': str(e)})
    try:
        with gpu_lock:
            nk = _stage('nukki')(image.file.read(), product_name=product_name, boxes=user_boxes)
            form = to_form_data(product_name, store_name, prompt, target, mood, ratio)
            vo = _stage('vlm')(nk, form)
            fo = _stage('generate')(nk, vo)
        print(f"[generate] {nk['name']} | 박스 {len(user_boxes or [])}개 | {time.time() - t0:.1f}초")
        return {'imageUrl': to_data_url(fo['final']), 'copy': to_copy(fo.get('ad_copy') or vo.get('ad_copy'), product_name)}
    except NukkiNotFound as e:
        return JSONResponse(status_code=422, content={'message': str(e)})
    except NotImplementedError as e:
        return JSONResponse(status_code=501, content={'message': str(e)})
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={'message': f'{type(e).__name__}: {e}'})

## 5-2. 서버 켜기
* 고정 주소로 서버를 열고, 웹 화면(Vercel)이 이 주소로 요청을 보냄
* 켜진 뒤 `주소/health`에서 세 단계(`nukki`, `vlm`, `generate`)가 모두 `true`인지 확인

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio
import socket
import nest_asyncio, uvicorn
from pyngrok import ngrok, conf
from google.colab import userdata

PORT = 8000
nest_asyncio.apply()

# ngrok 토큰과 고정 주소: 코랩 보안 비밀 NGROK_AUTHTOKEN, NGROK_DOMAIN
conf.get_default().auth_token = userdata.get('NGROK_AUTHTOKEN')
NGROK_DOMAIN = userdata.get('NGROK_DOMAIN')


def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0


# 서버: 이미 켜져 있으면 다시 켜지 않음
if not _port_in_use(PORT):
    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning'),
                     daemon=True).start()
    time.sleep(2)

# 터널: 이전 터널을 닫고 고정 주소로 다시 연결
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
try:
    tunnel = ngrok.connect(PORT, url=f'https://{NGROK_DOMAIN}')
except Exception:
    tunnel = ngrok.connect(PORT, domain=NGROK_DOMAIN)
PUBLIC_URL = tunnel.public_url
print('서버 주소:', PUBLIC_URL, '| 상태 확인:', PUBLIC_URL + '/health')